# Overview

This notebook writes a ton of plots. Simple ones that compare all columns to the response to detect relationships.

In [49]:
import polars as pl
import plotly.express as px
import polars.selectors as cs
import plotly.figure_factory as ff

data_directory = "../../data/energy"
filename = "s_energy.parquet"
df = pl.read_parquet(f"{data_directory}/{filename}")
# Filter for just 2024 to examine relationships
df = (
    df
    .filter(
        (pl.col("Year") == 2024) &
        (pl.col("EUI_WN").is_not_null())
    )
    .with_columns(
        (pl.col("Year") - pl.col("YearBuilt")).alias("BuildingAge")
    )
    .drop(["YearBuilt", "Year", "Years", "NumYears", "EUI", "Year2025"])
)

In [50]:
df.glimpse()

Rows: 3473
Columns: 10
$ ID                   <i64> 1, 2, 3, 5, 8, 9, 10, 11, 12, 15
$ BuildingName         <str> 'MAYFLOWER PARK HOTEL', 'PARAMOUNT HOTEL', 'WESTIN HOTEL (Parent Building)', 'HOTEL MAX', 'WARWICK SEATTLE HOTEL', 'WEST PRECINCT (SEATTLE POLICE)', 'CAMLIN WORLDMARK HOTEL', 'PARAMOUNT THEATRE', 'COURTYARD BY MARRIOTT - PIONEER SQ', 'HOTEL MONACO'
$ BuildingType         <str> 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential', 'NonResidential'
$ Neighborhood         <str> 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN', 'DOWNTOWN'
$ NumberofFloors       <i64> 12, 11, 41, 10, 18, 2, 11, 8, 15, 11
$ NumberofBuildings    <i64> 1, 1, 3, 1, 1, 1, 1, 1, 1, 1
$ PropertyGFABuildings <i64> 88434, 88502, 759392, 61320, 113580, 60090, 83008, 102761, 163984, 133884
$ EPAPropertyType      <str> 'Hotel', 'Hotel', 'Hotel'

In [51]:
df.describe()

statistic,ID,BuildingName,BuildingType,Neighborhood,NumberofFloors,NumberofBuildings,PropertyGFABuildings,EPAPropertyType,EUI_WN,BuildingAge
str,f64,str,str,str,f64,f64,f64,str,f64,f64
"""count""",3473.0,"""3473""","""3473""","""3473""",3473.0,3473.0,3473.0,"""3473""",3473.0,3473.0
"""null_count""",0.0,"""0""","""0""","""0""",0.0,0.0,0.0,"""0""",0.0,0.0
"""mean""",24772.641232,null,null,null,5.30262,1.269508,96302.809963,null,53.560669,48.60668
"""std""",14964.379585,null,null,null,6.082596,5.783382,295030.63309,null,146.682709,35.004734
"""min""",1.0,"""100 on 6th""","""Hospital""","""BALLARD""",0.0,0.0,16092.0,"""Adult Education""",0.1,1.0
"""25%""",20393.0,null,null,null,3.0,1.0,28431.0,null,27.1,20.0
"""50%""",23858.0,null,null,null,4.0,1.0,45000.0,null,35.5,41.0
"""75%""",27306.0,null,null,null,6.0,1.0,92051.0,null,53.0,68.0
"""max""",52046.0,"""citizenM South Lake Union Hote…","""University""","""SOUTHWEST""",76.0,300.0,1.5216474e7,"""Worship Facility""",5666.5,124.0


In [56]:
px.histogram(df, x="EUI_WN").show()
df = df.with_columns(pl.col("EUI_WN").log().alias("EUI_WN"))
px.histogram(df, x="EUI_WN").show()

In [57]:
numerical_columns = df.select(cs.numeric()).columns
numerical_columns

['ID',
 'NumberofFloors',
 'NumberofBuildings',
 'PropertyGFABuildings',
 'EUI_WN',
 'BuildingAge']

In [58]:
response = "EUI_WN"
columns = df.drop("ID", "BuildingName", response).columns
columns

['BuildingType',
 'Neighborhood',
 'NumberofFloors',
 'NumberofBuildings',
 'PropertyGFABuildings',
 'EPAPropertyType',
 'BuildingAge']

In [59]:
fig = px.box(
    df,
    y="BuildingType",
    x="EUI_WN",
    color="BuildingType"
)
fig.show()

In [60]:
boolean_columns = df.select(cs.boolean()).columns
categorical_columns = df.select(cs.string())
numerical_columns = df.select(cs.numeric()).columns
response = "EUI_WN"
columns = df.drop("ID", "BuildingName", response).columns
write = False
log_scale = False
for column in columns:

    if column in boolean_columns:
        fig = px.density_heatmap(
            df,
            x=response,
            y=column,
            text_auto=True,
            color_continuous_scale="Greys",
            title=f"Response breakdown for {column}"
        )

    elif column in categorical_columns:
        fig = px.box(
            df,
            y=column,
            x=response,
            color=column,
            log_x=log_scale,
            title=f"Response breakdown for {column}"
        )

    else:
        fig = px.scatter(
            df,
            x=response,
            y=column,
            log_x=log_scale,
            title=f"Response breakdown for {column}"
        )

    if write:
        fig.write_image(f"../../images/energy/{column}_vs_response.png")
    fig.show()